In [2]:
import sys
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import pandas as pd
import numpy as np

print("Imports done")

Imports done


In [3]:
df = pd.read_csv('../data/raw/subscription_data.csv')

print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nMissing values:")
print(df.isnull().sum())
df.head()

Shape: (332, 16)
Columns: ['company', 'issue_category', 'opening_date', 'issue_price', 'issue_amount_cr', 'qib_x', 'nii_x', 'retail_x', 'employees_x', 'others_x', 'shareholders_x', 'total_x', 'listing_date', 'open_price_listing', 'close_price_listing', 'gain_loss_pct']

Missing values:
company                  0
issue_category           0
opening_date             0
issue_price              0
issue_amount_cr         41
qib_x                    2
nii_x                    1
retail_x               331
employees_x            275
others_x               327
shareholders_x           3
total_x                  1
listing_date             0
open_price_listing       1
close_price_listing      1
gain_loss_pct          329
dtype: int64


,company,issue_category,opening_date,issue_price,issue_amount_cr,qib_x,nii_x,retail_x,employees_x,others_x,shareholders_x,total_x,listing_date,open_price_listing,close_price_listing,gain_loss_pct
Modern Diagnostic & Research Centre Ltd.,SME,31-Dec-2025,90,35.04,78.84,702.57,337.84,NaN,NaN,NaN,NaN,263.37,07-Jan-2026,99.50,94.99,5.54
E to E Transportation Infrastructure Ltd.,SME,26-Dec-2025,174,79.97,95.34,871.56,541.83,NaN,NaN,NaN,368.17,02-Jan-2026,330.60,327.80,88.39,NaN
Apollo Techno Industries Ltd.,SME,23-Dec-2025,130,45.55,11.12,98.46,48.13,NaN,NaN,NaN,37.20,31-Dec-2025,145.00,151.95,16.88,NaN
Bai Kakaji Polymers Ltd.,SME,23-Dec-2025,186,99.90,3.85,7.92,3.85,NaN,NaN,NaN,4.46,31-Dec-2025,190.00,188.20,1.18,NaN
Admach Systems Ltd.,SME,23-Dec-2025,239,40.47,1.57,7.37,3.85,NaN,NaN,NaN,3.84,31-Dec-2025,191.20,200.75,-16.00,NaN


In [4]:
# Fix 1: Calculate gain_loss_pct from prices we already have
# gain = (close_price_listing - issue_price) / issue_price * 100

df['issue_price'] = pd.to_numeric(df['issue_price'], errors='coerce')
df['close_price_listing'] = pd.to_numeric(df['close_price_listing'], errors='coerce')
df['open_price_listing'] = pd.to_numeric(df['open_price_listing'], errors='coerce')

df['listing_gain_pct'] = ((df['close_price_listing'] - df['issue_price']) / df['issue_price'] * 100).round(2)

print(f"listing_gain_pct non-null: {df['listing_gain_pct'].count()}")
print(f"Sample values: {df['listing_gain_pct'].dropna().head(10).tolist()}")

listing_gain_pct non-null: 331
Sample values: [171.09, 10.53, -62.94, -98.82, -139.54, -60.95, -47.51, -178.22, 172.38, -103.31]


In [5]:
# Fix 2: Fill missing subscription values with 0
# (means no data available for that category, not that it was 0)
for col in ['qib_x', 'nii_x', 'retail_x', 'total_x']:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

# Fix 3: Convert other numeric columns
for col in ['issue_price', 'issue_amount_cr']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Fix 4: Parse dates
df['listing_date'] = pd.to_datetime(df['listing_date'], errors='coerce')
df['listing_year'] = df['listing_date'].dt.year
df['listing_month'] = df['listing_date'].dt.month

print(f"Shape: {df.shape}")
print(f"\nMissing after fix:")
print(df[['qib_x','nii_x','retail_x','total_x','listing_gain_pct']].isnull().sum())

Shape: (332, 19)

Missing after fix:
qib_x               0
nii_x               0
retail_x            0
total_x             0
listing_gain_pct    1
dtype: int64


In [6]:
# Drop the 1 remaining NaN in target
df = df.dropna(subset=['listing_gain_pct'])

# Remove unrealistic values — real IPO gains/losses stay within -50% to +200%
before = len(df)
df = df[(df['listing_gain_pct'] >= -50) & (df['listing_gain_pct'] <= 200)]
after = len(df)
print(f"Removed {before - after} outlier rows")
print(f"Remaining: {after} rows")

print(f"\nlisting_gain_pct stats:")
print(df['listing_gain_pct'].describe())

Removed 279 outlier rows
Remaining: 52 rows

listing_gain_pct stats:
count     52.000000
mean      32.259808
std       72.634753
min      -49.780000
25%      -25.322500
50%        7.855000
75%       68.895000
max      199.190000
Name: listing_gain_pct, dtype: float64


In [7]:
# Encode SME vs Mainboard as 0/1
df['is_mainboard'] = (df['issue_category'] == 'Mainboard').astype(int)

# Select final columns for ML
df_clean = df[[
    'company', 'listing_date', 'listing_year', 'listing_month',
    'issue_price', 'issue_amount_cr', 'is_mainboard',
    'qib_x', 'nii_x', 'retail_x', 'total_x',
    'listing_gain_pct'
]].copy()

# Drop rows missing issue_amount_cr
df_clean = df_clean.dropna(subset=['issue_amount_cr'])

print(f"Final shape: {df_clean.shape}")
print(f"\nMissing values:")
print(df_clean.isnull().sum())

# Save
df_clean.to_csv('../data/processed/ipo_enriched.csv', index=False)
print("\nSaved to data/processed/ipo_enriched.csv")

Final shape: (46, 12)

Missing values:
company              0
listing_date        45
listing_year        45
listing_month       45
issue_price          0
issue_amount_cr      0
is_mainboard         0
qib_x                0
nii_x                0
retail_x             0
total_x              0
listing_gain_pct     0
dtype: int64

Saved to data/processed/ipo_enriched.csv


In [8]:
# The real issue: rows where close_price_listing is 0 or very small
# These are IPOs not yet listed — remove them properly

# Reload from scratch
df2 = pd.read_csv('../data/raw/subscription_data.csv')

# Convert prices
for col in ['issue_price', 'close_price_listing', 'open_price_listing']:
    df2[col] = pd.to_numeric(df2[col], errors='coerce')

# Remove rows where close price is 0 or missing (not yet listed)
df2 = df2[df2['close_price_listing'] > 0]
df2 = df2[df2['issue_price'] > 0]

# Recalculate gain
df2['listing_gain_pct'] = ((df2['close_price_listing'] - df2['issue_price']) / df2['issue_price'] * 100).round(2)

print(f"Rows after removing unlisted: {len(df2)}")
print(f"\nlisting_gain_pct stats:")
print(df2['listing_gain_pct'].describe())

Rows after removing unlisted: 199

listing_gain_pct stats:
count    199.000000
mean     -40.648090
std      103.450091
min     -100.000000
25%      -97.615000
50%      -81.730000
75%      -35.885000
max      536.280000
Name: listing_gain_pct, dtype: float64


In [9]:
# Fix subscription columns
for col in ['qib_x', 'nii_x', 'retail_x', 'total_x', 'issue_amount_cr']:
    df2[col] = pd.to_numeric(df2[col], errors='coerce').fillna(0)

# Fix dates — try multiple formats
df2['listing_date'] = pd.to_datetime(df2['listing_date'], dayfirst=True, errors='coerce')
df2['listing_year'] = df2['listing_date'].dt.year
df2['listing_month'] = df2['listing_date'].dt.month

# Encode category
df2['is_mainboard'] = (df2['issue_category'] == 'Mainboard').astype(int)

# Final clean dataset
df_final = df2[[
    'company', 'listing_date', 'listing_year', 'listing_month',
    'issue_price', 'issue_amount_cr', 'is_mainboard',
    'qib_x', 'nii_x', 'retail_x', 'total_x',
    'listing_gain_pct'
]].copy()

df_final = df_final.dropna(subset=['listing_gain_pct', 'listing_year'])

print(f"Final shape: {df_final.shape}")
print(f"Missing:\n{df_final.isnull().sum()}")
print(f"\nGain stats:\n{df_final['listing_gain_pct'].describe()}")

df_final.to_csv('../data/processed/ipo_enriched.csv', index=False)
print("\nSaved.")

Final shape: (2, 12)
Missing:
company             0
listing_date        0
listing_year        0
listing_month       0
issue_price         0
issue_amount_cr     0
is_mainboard        0
qib_x               0
nii_x               0
retail_x            0
total_x             0
listing_gain_pct    0
dtype: int64

Gain stats:
count      2.000000
mean     140.020000
std       43.939615
min      108.950000
25%      124.485000
50%      140.020000
75%      155.555000
max      171.090000
Name: listing_gain_pct, dtype: float64

Saved.


In [11]:
df3 = pd.read_csv('../data/raw/subscription_data.csv')

# Use the original gain_loss_pct column directly — don't recalculate
df3['gain_loss_pct'] = pd.to_numeric(df3['gain_loss_pct'], errors='coerce')

# Check how many have it
print(f"Total rows: {len(df3)}")
print(f"gain_loss_pct non-null: {df3['gain_loss_pct'].count()}")
print(f"\nSample values:")
print(df3['gain_loss_pct'].dropna().head(20).tolist())
print(f"\nStats:")
print(df3['gain_loss_pct'].describe())

Total rows: 332
gain_loss_pct non-null: 3

Sample values:
[5.54, 0.0, 4.88]

Stats:
count    3.000000
mean     3.473333
std      3.026043
min      0.000000
25%      2.440000
50%      4.880000
75%      5.210000
max      5.540000
Name: gain_loss_pct, dtype: float64
